# Sinhala Buddhist Corpus Builder - Document AI Batch Processing

This notebook extracts text from large Sinhala Buddhist PDFs using Google Cloud Document AI Batch Processing.

**Key Features:**
- Handles PDFs of any size (hundreds of pages)
- Batch processing via Google Cloud Storage
- Page-wise text file output (matching Vision API structure)
- Comprehensive checkpointing and resume capability
- Automatic cleanup (PDFs deleted from GCS after processing)
- One-by-one processing with immediate result retrieval

**Workflow:**
1. Google Drive (PDFs) → GCS (temporary)
2. Document AI Batch Processing
3. Parse results and save to Drive
4. Clean up GCS (delete temporary files)

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install --upgrade google-cloud-documentai google-cloud-storage google-auth tqdm -q

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import time
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import re

from google.cloud import documentai_v1 as documentai
from google.cloud import storage
from google.oauth2 import service_account
from google.api_core.client_options import ClientOptions
from tqdm.notebook import tqdm
import pandas as pd

## 2. Configuration

### Directory Configuration

In [ ]:
# Base directory configuration
BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/")
PDF_FOLDER = BASE_DIR / "data" / "raw" / "old_buddhist_books"

# Document AI extraction folders (separate from Vision API)
EXTRACTION_BASE = BASE_DIR / "data" / "docai_extractions"
RAW_TEXT_DIR = EXTRACTION_BASE / "1_raw_text"
CLEANED_TEXT_DIR = EXTRACTION_BASE / "2_cleaned_text"

# Logs and metadata
LOGS_DIR = EXTRACTION_BASE / "logs"
CHECKPOINT_FILE = LOGS_DIR / "processing_checkpoint.json"

# Create all directories
for directory in [RAW_TEXT_DIR, CLEANED_TEXT_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("✓ Directory structure created")
print(f"  PDFs from: {PDF_FOLDER}")
print(f"  Raw text to: {RAW_TEXT_DIR}")
print(f"  Cleaned text to: {CLEANED_TEXT_DIR}")
print(f"  Logs to: {LOGS_DIR}")
print(f"  Checkpoint: {CHECKPOINT_FILE}")

### Google Cloud Authentication

In [ ]:
# Path to your service account credentials JSON file
# TODO: Update this path to your actual credentials file location
CREDENTIALS_PATH = BASE_DIR / "credentials" / "service-account-key.json"

# Alternative: If your credentials are in a different location, update here:
# CREDENTIALS_PATH = "/content/drive/MyDrive/path/to/your-service-account.json"

if not CREDENTIALS_PATH.exists():
    print("⚠️ WARNING: Credentials file not found!")
    print(f"   Looking for: {CREDENTIALS_PATH}")
    print("\n   Please update CREDENTIALS_PATH to point to your service account JSON file.")
else:
    print("✓ Credentials file found")
    
    # Verify permissions
    print("\n⚠️ Ensure your service account has these roles:")
    print("   • Document AI Editor")
    print("   • Storage Object Admin")

### Document AI & GCS Configuration

**Required Information:**
- Project ID: Your Google Cloud project ID
- Location: Processor location (usually 'us' or 'eu')
- Processor ID: Your Document AI OCR processor ID
- GCS Bucket Name: Will be auto-created if doesn't exist

In [ ]:
# Google Cloud Configuration
# TODO: Update these with your actual values
PROJECT_ID = "your-project-id"  # e.g., "buddhist-nlp-project"
LOCATION = "us"  # e.g., "us" or "eu"
PROCESSOR_ID = "your-processor-id"  # e.g., "abc123def456"

# GCS Configuration
GCS_BUCKET_NAME = f"{PROJECT_ID}-docai-batch"  # Will be auto-created
GCS_INPUT_PREFIX = "input_pdfs/"  # Folder for PDFs
GCS_OUTPUT_PREFIX = "output_results/"  # Folder for results

# Processing configuration
POLL_INTERVAL = 30  # seconds between status checks
MAX_POLL_ATTEMPTS = 120  # max attempts (120 * 30s = 1 hour per PDF)
RETRY_ATTEMPTS = 3

print("Configuration:")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Processor: {PROCESSOR_ID}")
print(f"  GCS Bucket: {GCS_BUCKET_NAME}")
print(f"  Poll interval: {POLL_INTERVAL}s")

## 3. Initialize Clients

In [ ]:
def initialize_clients(credentials_path: Path, project_id: str, location: str):
    """
    Initialize Document AI and Cloud Storage clients.
    
    Returns:
        Tuple of (docai_client, storage_client, credentials)
    """
    # Load credentials
    credentials = service_account.Credentials.from_service_account_file(
        str(credentials_path),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )
    
    # Initialize Document AI client
    opts = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
    docai_client = documentai.DocumentProcessorServiceClient(
        client_options=opts,
        credentials=credentials
    )
    
    # Initialize Storage client
    storage_client = storage.Client(
        credentials=credentials,
        project=project_id
    )
    
    return docai_client, storage_client, credentials


# Initialize clients
try:
    docai_client, storage_client, credentials = initialize_clients(
        CREDENTIALS_PATH, PROJECT_ID, LOCATION
    )
    print("✓ Document AI client initialized")
    print("✓ Cloud Storage client initialized")
except Exception as e:
    print(f"✗ Error initializing clients: {e}")
    print("  Please check your credentials and configuration.")

## 4. GCS Bucket Setup

In [ ]:
def create_bucket_if_not_exists(storage_client: storage.Client, bucket_name: str, location: str) -> storage.Bucket:
    """
    Create GCS bucket if it doesn't exist.
    
    Args:
        storage_client: GCS client
        bucket_name: Name of bucket to create
        location: Bucket location
        
    Returns:
        Bucket object
    """
    try:
        bucket = storage_client.get_bucket(bucket_name)
        print(f"✓ Bucket '{bucket_name}' already exists")
    except Exception:
        # Bucket doesn't exist, create it
        bucket = storage_client.create_bucket(bucket_name, location=location)
        print(f"✓ Created bucket '{bucket_name}' in {location}")
    
    return bucket


# Create or get bucket
try:
    bucket = create_bucket_if_not_exists(storage_client, GCS_BUCKET_NAME, LOCATION)
    print(f"\n✓ Using bucket: gs://{GCS_BUCKET_NAME}")
except Exception as e:
    print(f"✗ Error with bucket: {e}")
    print("  Check if bucket name is unique globally or if you have permissions.")

## 5. Checkpoint Management

In [ ]:
class CheckpointManager:
    """
    Manages processing checkpoints for resume capability.
    
    Checkpoint structure:
    {
        "session_id": "timestamp",
        "processed_pdfs": {
            "pdf_name": {
                "status": "success|failed|processing",
                "pages": 123,
                "timestamp": "ISO datetime",
                "gcs_input_uri": "gs://...",
                "gcs_output_uri": "gs://...",
                "error": "error message if failed"
            }
        }
    }
    """
    
    def __init__(self, checkpoint_file: Path):
        self.checkpoint_file = checkpoint_file
        self.data = self._load()
        
    def _load(self) -> dict:
        """Load checkpoint from file."""
        if self.checkpoint_file.exists():
            with open(self.checkpoint_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        else:
            return {
                "session_id": datetime.now().strftime("%Y%m%d_%H%M%S"),
                "processed_pdfs": {}
            }
    
    def _save(self):
        """Save checkpoint to file."""
        with open(self.checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(self.data, f, indent=2, ensure_ascii=False)
    
    def is_processed(self, pdf_name: str) -> bool:
        """Check if PDF was successfully processed."""
        return (pdf_name in self.data["processed_pdfs"] and 
                self.data["processed_pdfs"][pdf_name]["status"] == "success")
    
    def mark_processing(self, pdf_name: str, gcs_input_uri: str, gcs_output_uri: str):
        """Mark PDF as currently processing."""
        self.data["processed_pdfs"][pdf_name] = {
            "status": "processing",
            "timestamp": datetime.now().isoformat(),
            "gcs_input_uri": gcs_input_uri,
            "gcs_output_uri": gcs_output_uri
        }
        self._save()
    
    def mark_success(self, pdf_name: str, pages: int):
        """Mark PDF as successfully processed."""
        if pdf_name in self.data["processed_pdfs"]:
            self.data["processed_pdfs"][pdf_name]["status"] = "success"
            self.data["processed_pdfs"][pdf_name]["pages"] = pages
            self.data["processed_pdfs"][pdf_name]["completed_at"] = datetime.now().isoformat()
            self._save()
    
    def mark_failed(self, pdf_name: str, error: str):
        """Mark PDF as failed."""
        if pdf_name in self.data["processed_pdfs"]:
            self.data["processed_pdfs"][pdf_name]["status"] = "failed"
            self.data["processed_pdfs"][pdf_name]["error"] = error
            self.data["processed_pdfs"][pdf_name]["failed_at"] = datetime.now().isoformat()
            self._save()
    
    def get_summary(self) -> str:
        """Get summary of processed PDFs."""
        processed = self.data["processed_pdfs"]
        total = len(processed)
        success = sum(1 for p in processed.values() if p["status"] == "success")
        failed = sum(1 for p in processed.values() if p["status"] == "failed")
        processing = sum(1 for p in processed.values() if p["status"] == "processing")
        total_pages = sum(p.get("pages", 0) for p in processed.values() if p["status"] == "success")
        
        return f"""
Checkpoint Summary:
─────────────────────────────────
Total PDFs tracked: {total}
Successfully processed: {success}
Failed: {failed}
Currently processing: {processing}
Total pages extracted: {total_pages}
Session ID: {self.data['session_id']}
"""


# Initialize checkpoint manager
checkpoint = CheckpointManager(CHECKPOINT_FILE)
print(checkpoint.get_summary())

## 6. GCS Upload/Download Functions

In [ ]:
def upload_pdf_to_gcs(
    storage_client: storage.Client,
    bucket_name: str,
    pdf_path: Path,
    gcs_prefix: str
) -> str:
    """
    Upload PDF from Drive to GCS.
    
    Args:
        storage_client: GCS client
        bucket_name: Target bucket name
        pdf_path: Local path to PDF file
        gcs_prefix: GCS folder prefix
        
    Returns:
        GCS URI (gs://bucket/path/to/file.pdf)
    """
    bucket = storage_client.bucket(bucket_name)
    
    # Create blob name (keep original filename)
    blob_name = f"{gcs_prefix}{pdf_path.name}"
    blob = bucket.blob(blob_name)
    
    # Upload file
    blob.upload_from_filename(str(pdf_path))
    
    gcs_uri = f"gs://{bucket_name}/{blob_name}"
    return gcs_uri


def delete_from_gcs(
    storage_client: storage.Client,
    gcs_uri: str
) -> None:
    """
    Delete file or folder from GCS.
    
    Args:
        storage_client: GCS client
        gcs_uri: Full GCS URI (gs://bucket/path)
    """
    # Parse URI
    if not gcs_uri.startswith("gs://"):
        return
    
    parts = gcs_uri[5:].split("/", 1)
    bucket_name = parts[0]
    blob_path = parts[1] if len(parts) > 1 else ""
    
    bucket = storage_client.bucket(bucket_name)
    
    # If it's a folder, delete all blobs with that prefix
    if blob_path.endswith("/") or not blob_path:
        blobs = list(bucket.list_blobs(prefix=blob_path))
        for blob in blobs:
            blob.delete()
    else:
        # Single file
        blob = bucket.blob(blob_path)
        if blob.exists():
            blob.delete()


def download_results_from_gcs(
    storage_client: storage.Client,
    gcs_output_uri: str
) -> List[bytes]:
    """
    Download all JSON result files from GCS output folder.
    
    Args:
        storage_client: GCS client
        gcs_output_uri: GCS URI of output folder
        
    Returns:
        List of JSON content as bytes
    """
    # Parse URI
    parts = gcs_output_uri[5:].split("/", 1)
    bucket_name = parts[0]
    prefix = parts[1] if len(parts) > 1 else ""
    
    bucket = storage_client.bucket(bucket_name)
    
    # List all JSON files in output folder
    blobs = list(bucket.list_blobs(prefix=prefix))
    json_blobs = [b for b in blobs if b.name.endswith(".json")]
    
    # Download each JSON file
    results = []
    for blob in json_blobs:
        content = blob.download_as_bytes()
        results.append(content)
    
    return results


print("✓ GCS functions defined")

## 7. Document AI Batch Processing Functions

In [ ]:
def get_processor_name(project_id: str, location: str, processor_id: str) -> str:
    """Construct full processor resource name."""
    return f"projects/{project_id}/locations/{location}/processors/{processor_id}"


def submit_batch_process_request(
    client: documentai.DocumentProcessorServiceClient,
    processor_name: str,
    gcs_input_uri: str,
    gcs_output_uri: str
) -> documentai.BatchProcessMetadata:
    """
    Submit a batch processing request to Document AI.
    
    Args:
        client: Document AI client
        processor_name: Full processor resource name
        gcs_input_uri: GCS URI of input PDF
        gcs_output_uri: GCS URI for output folder
        
    Returns:
        Operation object for tracking
    """
    # Configure input
    gcs_document = documentai.GcsDocument(
        gcs_uri=gcs_input_uri,
        mime_type="application/pdf"
    )
    
    gcs_documents = documentai.GcsDocuments(documents=[gcs_document])
    input_config = documentai.BatchDocumentsInputConfig(gcs_documents=gcs_documents)
    
    # Configure output
    gcs_output_config = documentai.DocumentOutputConfig.GcsOutputConfig(
        gcs_uri=gcs_output_uri
    )
    output_config = documentai.DocumentOutputConfig(gcs_output_config=gcs_output_config)
    
    # Create request
    request = documentai.BatchProcessRequest(
        name=processor_name,
        input_documents=input_config,
        document_output_config=output_config,
    )
    
    # Submit
    operation = client.batch_process_documents(request)
    
    return operation


def wait_for_operation(
    operation,
    poll_interval: int = 30,
    max_attempts: int = 120
) -> bool:
    """
    Wait for batch processing operation to complete.
    
    Args:
        operation: Long-running operation object
        poll_interval: Seconds between status checks
        max_attempts: Maximum polling attempts
        
    Returns:
        True if successful, False if failed or timed out
    """
    print(f"  Waiting for processing to complete...")
    
    for attempt in range(max_attempts):
        if operation.done():
            if operation.exception():
                print(f"  ✗ Operation failed: {operation.exception()}")
                return False
            else:
                print(f"  ✓ Operation completed successfully")
                return True
        
        # Show progress every 5 attempts (2.5 minutes)
        if attempt % 5 == 0 and attempt > 0:
            elapsed = attempt * poll_interval
            print(f"  Still processing... ({elapsed}s elapsed)")
        
        time.sleep(poll_interval)
    
    print(f"  ⚠️ Timeout after {max_attempts * poll_interval}s")
    return False


print("✓ Batch processing functions defined")

## 8. Result Parsing Functions

In [ ]:
def parse_batch_results(result_bytes: bytes) -> documentai.Document:
    """
    Parse Document AI batch result JSON.
    
    Args:
        result_bytes: Raw JSON bytes from GCS
        
    Returns:
        Parsed Document object
    """
    document = documentai.Document.from_json(result_bytes)
    return document


def extract_text_by_page(document: documentai.Document) -> List[str]:
    """
    Extract text from Document AI response, organized by page.
    
    Args:
        document: Processed document from Document AI
        
    Returns:
        List of text strings, one per page
    """
    page_texts = []
    
    for page in document.pages:
        page_text = ""
        
        # Extract text from paragraphs (maintains reading order)
        if hasattr(page, 'paragraphs') and page.paragraphs:
            for paragraph in page.paragraphs:
                text_segment = paragraph.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + "\n\n"
        
        # Fallback: If no paragraphs, use lines
        elif hasattr(page, 'lines') and page.lines:
            for line in page.lines:
                text_segment = line.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + "\n"
        
        # Fallback: If no structured layout, use tokens
        elif hasattr(page, 'tokens') and page.tokens:
            for token in page.tokens:
                text_segment = token.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + " "
            page_text = page_text.strip() + "\n"
        
        page_texts.append(page_text.strip())
    
    return page_texts


def save_page_texts(page_texts: List[str], output_dir: Path, pdf_name: str) -> None:
    """
    Save extracted text to individual page files.
    
    Creates files named: {pdf_name}/page_001.txt, page_002.txt, etc.
    
    Args:
        page_texts: List of text strings, one per page
        output_dir: Base output directory (RAW_TEXT_DIR or CLEANED_TEXT_DIR)
        pdf_name: Name of the PDF (without extension)
    """
    # Create subdirectory for this PDF
    pdf_output_dir = output_dir / pdf_name
    pdf_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save each page
    for page_num, page_text in enumerate(page_texts, start=1):
        page_filename = f"page_{page_num:03d}.txt"
        page_file_path = pdf_output_dir / page_filename
        
        with open(page_file_path, 'w', encoding='utf-8') as f:
            f.write(page_text)


print("✓ Result parsing functions defined")

## 9. Text Cleaning Functions

In [ ]:
def clean_text(text: str) -> str:
    """
    Apply basic text cleaning operations.
    
    Operations:
    - Remove excessive whitespace
    - Normalize line breaks
    - Preserve Sinhala unicode
    
    Args:
        text: Raw text to clean
        
    Returns:
        Cleaned text
    """
    if not text:
        return ""
    
    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)  # Multiple spaces/tabs to single space
    text = re.sub(r'\n\s*\n', '\n\n', text)  # Multiple blank lines to double line break
    
    # Remove leading/trailing whitespace from each line
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    
    # Remove excessive blank lines
    text = re.sub(r'\n\n+', '\n\n', text)
    
    return text.strip()


def clean_and_save_texts(raw_text_dir: Path, cleaned_text_dir: Path, pdf_name: str) -> None:
    """
    Clean all page texts for a PDF and save to cleaned directory.
    
    Args:
        raw_text_dir: Directory containing raw text files
        cleaned_text_dir: Directory to save cleaned text files
        pdf_name: Name of the PDF (without extension)
    """
    raw_pdf_dir = raw_text_dir / pdf_name
    cleaned_pdf_dir = cleaned_text_dir / pdf_name
    cleaned_pdf_dir.mkdir(parents=True, exist_ok=True)
    
    # Process each page file
    for page_file in sorted(raw_pdf_dir.glob("page_*.txt")):
        with open(page_file, 'r', encoding='utf-8') as f:
            raw_text = f.read()
        
        cleaned_text = clean_text(raw_text)
        
        output_file = cleaned_pdf_dir / page_file.name
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(cleaned_text)


print("✓ Text cleaning functions defined")

## 10. Main Processing Pipeline

In [ ]:
def process_single_pdf_batch(
    pdf_path: Path,
    docai_client: documentai.DocumentProcessorServiceClient,
    storage_client: storage.Client,
    processor_name: str,
    bucket_name: str,
    gcs_input_prefix: str,
    gcs_output_prefix: str,
    raw_text_dir: Path,
    cleaned_text_dir: Path,
    checkpoint: CheckpointManager,
    poll_interval: int,
    max_poll_attempts: int
) -> Tuple[str, int]:
    """
    Process a single PDF through the complete batch pipeline.
    
    Pipeline:
    1. Upload PDF to GCS
    2. Submit batch processing request
    3. Wait for completion
    4. Download and parse results
    5. Extract text by page and save
    6. Clean text and save
    7. Delete PDF from GCS (cleanup)
    8. Update checkpoint
    
    Args:
        pdf_path: Path to PDF file
        docai_client: Document AI client
        storage_client: GCS client
        processor_name: Full processor resource name
        bucket_name: GCS bucket name
        gcs_input_prefix: GCS input folder prefix
        gcs_output_prefix: GCS output folder prefix
        raw_text_dir: Directory for raw text output
        cleaned_text_dir: Directory for cleaned text output
        checkpoint: Checkpoint manager
        poll_interval: Seconds between status checks
        max_poll_attempts: Maximum polling attempts
        
    Returns:
        Tuple of (status, page_count)
    """
    pdf_name = pdf_path.stem
    start_time = time.time()
    
    try:
        # Step 1: Upload PDF to GCS
        print(f"  Uploading to GCS...")
        gcs_input_uri = upload_pdf_to_gcs(
            storage_client=storage_client,
            bucket_name=bucket_name,
            pdf_path=pdf_path,
            gcs_prefix=gcs_input_prefix
        )
        print(f"  ✓ Uploaded: {gcs_input_uri}")
        
        # Prepare output URI
        gcs_output_uri = f"gs://{bucket_name}/{gcs_output_prefix}{pdf_name}/"
        
        # Update checkpoint
        checkpoint.mark_processing(pdf_name, gcs_input_uri, gcs_output_uri)
        
        # Step 2: Submit batch processing request
        print(f"  Submitting batch processing request...")
        operation = submit_batch_process_request(
            client=docai_client,
            processor_name=processor_name,
            gcs_input_uri=gcs_input_uri,
            gcs_output_uri=gcs_output_uri
        )
        print(f"  ✓ Request submitted")
        
        # Step 3: Wait for completion
        success = wait_for_operation(
            operation=operation,
            poll_interval=poll_interval,
            max_attempts=max_poll_attempts
        )
        
        if not success:
            raise Exception("Batch processing failed or timed out")
        
        # Step 4: Download results from GCS
        print(f"  Downloading results...")
        result_bytes_list = download_results_from_gcs(
            storage_client=storage_client,
            gcs_output_uri=gcs_output_uri
        )
        
        if not result_bytes_list:
            raise Exception("No results found in GCS output folder")
        
        print(f"  ✓ Downloaded {len(result_bytes_list)} result file(s)")
        
        # Step 5: Parse results and extract text
        print(f"  Parsing results and extracting text...")
        all_page_texts = []
        
        for result_bytes in result_bytes_list:
            document = parse_batch_results(result_bytes)
            page_texts = extract_text_by_page(document)
            all_page_texts.extend(page_texts)
        
        page_count = len(all_page_texts)
        print(f"  ✓ Extracted {page_count} pages")
        
        # Step 6: Save raw text
        print(f"  Saving raw text files...")
        save_page_texts(all_page_texts, raw_text_dir, pdf_name)
        print(f"  ✓ Saved to {raw_text_dir / pdf_name}")
        
        # Step 7: Clean and save cleaned text
        print(f"  Cleaning and saving cleaned text...")
        clean_and_save_texts(raw_text_dir, cleaned_text_dir, pdf_name)
        print(f"  ✓ Saved to {cleaned_text_dir / pdf_name}")
        
        # Step 8: Cleanup - Delete from GCS
        print(f"  Cleaning up GCS...")
        delete_from_gcs(storage_client, gcs_input_uri)  # Delete input PDF
        delete_from_gcs(storage_client, gcs_output_uri)  # Delete output folder
        print(f"  ✓ Deleted temporary files from GCS")
        
        # Update checkpoint
        checkpoint.mark_success(pdf_name, page_count)
        
        processing_time = time.time() - start_time
        print(f"  ✓ Total processing time: {processing_time:.2f}s ({processing_time/60:.2f} min)")
        
        return "success", page_count
        
    except Exception as e:
        # Log failure
        error_msg = str(e)
        print(f"  ✗ Error: {error_msg}")
        
        # Update checkpoint
        checkpoint.mark_failed(pdf_name, error_msg)
        
        # Try to cleanup GCS on failure
        try:
            if 'gcs_input_uri' in locals():
                delete_from_gcs(storage_client, gcs_input_uri)
            if 'gcs_output_uri' in locals():
                delete_from_gcs(storage_client, gcs_output_uri)
        except:
            pass
        
        return "failed", 0


print("✓ Main processing pipeline defined")

## 11. Run Batch Processing

**Before running:**
1. ✓ Mount Google Drive
2. ✓ Update credentials path
3. ✓ Update PROJECT_ID, LOCATION, PROCESSOR_ID
4. ✓ Verify service account has required permissions
5. ✓ Verify PDF folder contains files

In [ ]:
# Pre-flight checks
print("Pre-flight checks:")
print(f"  PDF folder exists: {PDF_FOLDER.exists()}")
pdf_files = list(PDF_FOLDER.glob('*.pdf'))
print(f"  PDF count: {len(pdf_files)}")
print(f"  Credentials exist: {CREDENTIALS_PATH.exists()}")
print(f"  Output directories ready: {RAW_TEXT_DIR.exists()}")
print(f"  GCS bucket ready: gs://{GCS_BUCKET_NAME}")
print(f"\nDocument AI config:")
print(f"  Project: {PROJECT_ID}")
print(f"  Processor: {PROCESSOR_ID}")
print(f"\nCheckpoint status:")
print(checkpoint.get_summary())

In [ ]:
# Build processor name
PROCESSOR_NAME = get_processor_name(PROJECT_ID, LOCATION, PROCESSOR_ID)
print(f"Full processor name: {PROCESSOR_NAME}")

In [ ]:
# Get list of PDFs to process
pdf_files = sorted(PDF_FOLDER.glob("*.pdf"))

# Filter out already processed
pdfs_to_process = [
    pdf for pdf in pdf_files 
    if not checkpoint.is_processed(pdf.stem)
]

print(f"\n📚 Total PDFs found: {len(pdf_files)}")
print(f"📚 Already processed: {len(pdf_files) - len(pdfs_to_process)}")
print(f"📚 To process: {len(pdfs_to_process)}\n")

if not pdfs_to_process:
    print("✓ All PDFs already processed!")
else:
    print("PDFs to process:")
    for i, pdf in enumerate(pdfs_to_process[:10], 1):  # Show first 10
        print(f"  {i}. {pdf.name}")
    if len(pdfs_to_process) > 10:
        print(f"  ... and {len(pdfs_to_process) - 10} more")

In [ ]:
# Process all PDFs one by one
if pdfs_to_process:
    print("\n" + "="*70)
    print("STARTING BATCH PROCESSING")
    print("="*70 + "\n")
    
    total_pages = 0
    total_time = 0
    successful = 0
    failed = 0
    
    for idx, pdf_path in enumerate(pdfs_to_process, 1):
        print(f"\n[{idx}/{len(pdfs_to_process)}] Processing: {pdf_path.name}")
        print("-" * 70)
        
        start_time = time.time()
        
        status, page_count = process_single_pdf_batch(
            pdf_path=pdf_path,
            docai_client=docai_client,
            storage_client=storage_client,
            processor_name=PROCESSOR_NAME,
            bucket_name=GCS_BUCKET_NAME,
            gcs_input_prefix=GCS_INPUT_PREFIX,
            gcs_output_prefix=GCS_OUTPUT_PREFIX,
            raw_text_dir=RAW_TEXT_DIR,
            cleaned_text_dir=CLEANED_TEXT_DIR,
            checkpoint=checkpoint,
            poll_interval=POLL_INTERVAL,
            max_poll_attempts=MAX_POLL_ATTEMPTS
        )
        
        elapsed = time.time() - start_time
        
        if status == "success":
            print(f"\n✓ SUCCESS: {page_count} pages extracted in {elapsed:.2f}s")
            successful += 1
            total_pages += page_count
        else:
            print(f"\n✗ FAILED after {elapsed:.2f}s")
            failed += 1
        
        total_time += elapsed
        
        # Show progress summary
        print(f"\nProgress: {successful} successful, {failed} failed, {len(pdfs_to_process) - idx} remaining")
        print("=" * 70)
    
    # Final summary
    print("\n" + "="*70)
    print("BATCH PROCESSING COMPLETE")
    print("="*70)
    print(f"\nTotal PDFs processed: {len(pdfs_to_process)}")
    print(f"Successful: {successful}")
    print(f"Failed: {failed}")
    print(f"Total pages extracted: {total_pages}")
    print(f"Total time: {total_time:.2f}s ({total_time/60:.2f} minutes)")
    if successful > 0:
        print(f"Average time per PDF: {total_time/successful:.2f}s")
        print(f"Average pages per PDF: {total_pages/successful:.1f}")

## 12. View Results and Statistics

In [ ]:
# Display checkpoint summary
print(checkpoint.get_summary())

In [ ]:
def get_extraction_statistics(raw_text_dir: Path) -> pd.DataFrame:
    """
    Generate statistics about extracted text.
    
    Returns:
        DataFrame with per-PDF statistics
    """
    stats = []
    
    for pdf_dir in sorted(raw_text_dir.iterdir()):
        if not pdf_dir.is_dir():
            continue
        
        pdf_name = pdf_dir.name
        page_files = list(pdf_dir.glob("page_*.txt"))
        
        # Calculate total characters
        total_chars = 0
        for page_file in page_files:
            with open(page_file, 'r', encoding='utf-8') as f:
                total_chars += len(f.read())
        
        stats.append({
            "PDF Name": pdf_name,
            "Pages": len(page_files),
            "Total Characters": total_chars,
            "Avg Chars/Page": total_chars // len(page_files) if page_files else 0
        })
    
    df = pd.DataFrame(stats)
    return df


# Generate and display statistics
if list(RAW_TEXT_DIR.iterdir()):
    stats_df = get_extraction_statistics(RAW_TEXT_DIR)
    print("\n" + "="*60)
    print("EXTRACTION STATISTICS")
    print("="*60)
    print(stats_df.to_string(index=False))
    print("\nTotals:")
    print(f"  Total PDFs: {len(stats_df)}")
    print(f"  Total Pages: {stats_df['Pages'].sum()}")
    print(f"  Total Characters: {stats_df['Total Characters'].sum():,}")
else:
    print("No extracted text files found yet.")

In [ ]:
# Sample extracted text (first page of first PDF)
sample_pdfs = sorted([d for d in RAW_TEXT_DIR.iterdir() if d.is_dir()])
if sample_pdfs:
    sample_pdf_dir = sample_pdfs[0]
    sample_pages = list(sample_pdf_dir.glob("page_*.txt"))
    
    if sample_pages:
        sample_page = sample_pages[0]
        
        print("\n" + "="*60)
        print(f"SAMPLE EXTRACTED TEXT")
        print(f"PDF: {sample_pdf_dir.name}")
        print(f"Page: {sample_page.name}")
        print("="*60)
        
        with open(sample_page, 'r', encoding='utf-8') as f:
            sample_text = f.read()
        
        # Display first 500 characters
        print(sample_text[:500])
        print("\n[...truncated...]" if len(sample_text) > 500 else "")
else:
    print("No sample text available yet.")

## 13. Manual Operations (Optional)

### Retry Failed PDFs

If any PDFs failed, you can retry them by updating the checkpoint status and re-running the processing cell.

In [ ]:
# View failed PDFs
failed_pdfs = [
    (name, info["error"]) 
    for name, info in checkpoint.data["processed_pdfs"].items() 
    if info["status"] == "failed"
]

if failed_pdfs:
    print(f"\nFailed PDFs ({len(failed_pdfs)}):")
    for name, error in failed_pdfs:
        print(f"  • {name}")
        print(f"    Error: {error[:100]}..." if len(error) > 100 else f"    Error: {error}")
else:
    print("No failed PDFs!")

In [ ]:
# To retry a specific failed PDF, remove it from checkpoint
# Uncomment and modify the following:

# pdf_to_retry = "your-pdf-name-here"  # Without .pdf extension
# if pdf_to_retry in checkpoint.data["processed_pdfs"]:
#     del checkpoint.data["processed_pdfs"][pdf_to_retry]
#     checkpoint._save()
#     print(f"Removed {pdf_to_retry} from checkpoint. Re-run processing cell to retry.")

### Clean Up GCS Bucket (Optional)

After processing is complete, you can verify the GCS bucket is clean (all temporary files deleted).

In [ ]:
# List remaining files in GCS bucket
bucket = storage_client.bucket(GCS_BUCKET_NAME)
blobs = list(bucket.list_blobs())

if blobs:
    print(f"⚠️ Warning: {len(blobs)} file(s) still in GCS bucket:")
    for blob in blobs[:20]:  # Show first 20
        print(f"  • {blob.name}")
    if len(blobs) > 20:
        print(f"  ... and {len(blobs) - 20} more")
    print("\nThese may be from failed processing. You can manually delete them if needed.")
else:
    print("✓ GCS bucket is clean! All temporary files have been deleted.")

## Summary

**What this notebook does:**
1. ✓ Processes large PDFs (hundreds of pages) using Document AI Batch Processing
2. ✓ Uploads PDFs from Drive to GCS temporarily
3. ✓ Submits batch requests and waits for completion
4. ✓ Downloads and parses results
5. ✓ Extracts text page-by-page (matching Vision API structure)
6. ✓ Saves both raw and cleaned versions
7. ✓ Automatically cleans up GCS (deletes temporary files)
8. ✓ Provides robust checkpointing and resume capability
9. ✓ Processes one-by-one with immediate results

**Output structure:**
```
data/docai_extractions/
├── 1_raw_text/
│   ├── pdf_name_1/
│   │   ├── page_001.txt
│   │   ├── page_002.txt
│   │   └── ...
├── 2_cleaned_text/
│   ├── pdf_name_1/
│   │   ├── page_001.txt
│   │   └── ...
└── logs/
    └── processing_checkpoint.json
```

**Resume capability:**
- If processing is interrupted, simply re-run the processing cell
- The checkpoint system tracks completed PDFs
- Only unprocessed PDFs will be processed

**Cost optimization:**
- PDFs are automatically deleted from GCS after processing
- Only Drive storage is used for permanent files
- GCS is used only as temporary staging area

**Next steps:**
1. Compare Document AI vs Vision API outputs for quality
2. Further text cleaning and normalization if needed
3. Corpus preparation for pretraining/fine-tuning